# RF-DETR trained on native-resolution TILES (flatbug's approach)

**Why.** flatbug never resizes: its pipeline is `RandomCrop → CenterCrop(1024)`, so insects are always seen at **1:1 pixel scale**, and it tiles again at inference. RF-DETR instead **square-resizes** each ~1536×1150 tray to 800×800, shrinking every insect to ~0.52× horizontally. That lands exactly where RF-DETR loses — high-IoU localisation (mAP@75 0.340 vs flatbug's 0.468) — while leaving loose-IoU detection roughly equal (mAP@50 0.641 vs 0.627).

This notebook removes that confound: cut the trays into **800 px tiles**, train RF-DETR on them at `RESOLUTION=800` (so tiles are fed **1:1, no downscaling**), then tile at inference and merge back to full-tray predictions for a number that is directly comparable to your other runs.

**Steps:** export tiled COCO → verify → train → tiled inference + merge → score.

> The tiling and merge logic is unit-tested: 6 tiles give 100% pixel coverage of a > 1536×1150 tray, every ground-truth box survives intact in at least one tile, and a > perfect-detector round-trip recovers **60/60** objects after merging.

> **Set `GPU_ID` to a free GPU** — this is meant to run alongside the 1024 experiments.


In [ ]:
# ============================================================
# SEPTEMBER RUN — RF-DETR, 1536 px tiles fed 1:1
# ============================================================
# Regenerated dataset, trained at full resolution. What changed since August:
#
#   * GROUND TRUTH boxes are the insect, not the rotated canvas. The old boxes were
#     1.295x too large on a side (4/pi for a uniform angle), so ~40% of every box was
#     tray. Every AP number from before was measured against inflated targets.
#   * TRAYS are 3473x2769 = 10.85 px/mm, matching the real 3DTRAB trays exactly.
#     Previously 2644x2108 = 8.26 px/mm, so synthetic insects were 24% too small.
#   * TAXONOMY is 22 classes, not 23. Siphonaptera and Raphidioptera dropped;
#     Diplopoda -> Polyxenida, Chilopoda -> Lithobiomorpha + Scutigeromorpha.
#   * DATA is synthetic + real 3DTRAB mixed, group-disjoint on the real side, with
#     the real half oversampled to ~5% of the training signal.
#
# Evaluate on 3dtrab_coco_TESTSPLIT.json, NEVER the full 3DTRAB — roughly half of
# those photographs are in this training set.

# ============================================================
# CONFIG  <- edit GPU_ID (one per parallel run) and paths
# ============================================================
import os
GPU_ID = "2"                       # <-- 0/1/2/3, must differ from the other runs
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
from pathlib import Path

BASE_DIR    = Path(os.environ.get("MA_BASE", "/data/MA_Atul_Ares"))   # <-- Ares path
SRC_COCO    = BASE_DIR / "insect_dataset_mixed"          # full trays, 3473x2769
TILED_COCO  = BASE_DIR / "insect_dataset_mixed_tiled1536"
OUTPUT_DIR  = str(BASE_DIR / "sept_run" / "rfdetr_tiled_1536")

# ── tiling ──
TILE        = 1536    # == RESOLUTION, so tiles are fed 1:1 with NO resize.
                      # This is the whole point: an insect keeps its native 70 px
                      # (synthetic) / 131 px (real) instead of being shrunk to 31/58.
                      # At 1536/256 a 3473x2769 tray yields 6 tiles; at 800/200 it
                      # would yield 30, i.e. 360,000 training tiles.
OVERLAP     = 256     # every insect is complete in >=1 tile (largest real box ~180 px)
MIN_VIS     = 0.85    # keep a box only if >=85% of its area survives the crop
MIN_SIZE    = 4       # drop degenerate boxes (px)
MAX_TILES_PER_SPLIT = None   # e.g. 40000 to cap the train split; None = all

# ── inference merge ──
MERGE_IOU   = 0.6     # cross-tile duplicates are IoU~1.0; genuinely overlapping
                      # insects are usually <0.6, so this keeps both
EDGE_MARGIN = 16      # detections touching a tile border are demoted...
EDGE_FACTOR = 0.85    # ...by this factor, so the complete copy from the
                      # neighbouring tile wins NMS
INFER_CONF  = 0.05    # low for scoring; filter later for pictures

# ── model / training (identical to the 800px baseline apart from the data) ──
MODEL_VARIANT = "RFDETRLarge"
RESOLUTION    = 1536  # == TILE, so no resize happens
EPOCHS                   = 30    # ~6 tiles per tray, so an epoch is ~6x longer
BATCH_SIZE               = 4
GRAD_ACCUM_STEPS         = 4
LR                       = 1e-4
LR_ENCODER               = 1.5e-4
LR_DROP                  = 25
WARMUP_EPOCHS            = 5
WEIGHT_DECAY             = 1e-4
MULTI_SCALE              = True
USE_EMA                  = False
CHECKPOINT_INTERVAL      = 5
EVAL_INTERVAL            = 1
RUN_TEST                 = True
EARLY_STOPPING           = True
EARLY_STOPPING_PATIENCE  = 10
EARLY_STOPPING_MIN_DELTA = 0.001
TENSORBOARD              = True
WANDB                    = False
NUM_WORKERS              = 8
PROGRESS_BAR             = "rich"
NUM_GPUS                 = 1
DDP_STRATEGY             = "auto"

CLASS_NAMES = [
    # Insecta — 17 orders.  Siphonaptera (6 specimens) and Raphidioptera (2) were
    # dropped from the taxonomy; their specimens are on the prune blocklist so they
    # never reached a tray.
    "Lepidoptera", "Psocodea", "Odonata", "Zoraptera",
    "Phasmatodea", "Zygentoma", "Thysanoptera", "Strepsiptera",
    "Archaeognatha", "Dermaptera",
    "Diptera", "Coleoptera", "Hymenoptera", "Hemiptera",
    "Orthoptera", "Trichoptera", "Neuroptera",
    # Arachnida — 2 orders
    "Araneae", "Opiliones",
    # Myriapods at ORDER level.  Previously the class-level labels "Diplopoda" and
    # "Chilopoda", each covering morphologically unrelated orders — which is why both
    # scored badly (0.19 and 0.34).
    "Polyxenida", "Lithobiomorpha", "Scutigeromorpha",
]
assert (SRC_COCO / "train" / "_annotations.coco.json").exists(), f"missing {SRC_COCO}"
print(f"GPU {GPU_ID} | tiles {TILE}px overlap {OVERLAP} | {SRC_COCO} -> {TILED_COCO}")
print(f"train output -> {OUTPUT_DIR}")


In [ ]:
# ── Tiling core (grid, box tiling, NMS, edge penalty, merge) ──
import json, math
import numpy as np
from pathlib import Path


def tile_grid(W, H, tile, overlap):
    """Top-left corners covering WxH with `tile` windows and `overlap` px overlap.
    Last row/col is flush to the edge so nothing is missed."""
    step = max(1, tile - overlap)
    xs = list(range(0, max(1, W - tile + 1), step))
    ys = list(range(0, max(1, H - tile + 1), step))
    if not xs or xs[-1] != max(0, W - tile): xs.append(max(0, W - tile))
    if not ys or ys[-1] != max(0, H - tile): ys.append(max(0, H - tile))
    return sorted(set(xs)), sorted(set(ys))


def tile_boxes(boxes, x0, y0, tile, W, H, min_vis=0.85, min_size=4):
    """Clip xyxy boxes into the tile at (x0,y0). Returns (kept_boxes_local, kept_idx).
    A box is kept only if >= min_vis of its area survives the crop, so partial
    insects at tile edges are not learned as whole objects (they are complete in
    the overlapping neighbour tile)."""
    if len(boxes) == 0:
        return np.zeros((0, 4)), []
    b = np.asarray(boxes, float)
    x1 = np.clip(b[:, 0], x0, x0 + tile); y1 = np.clip(b[:, 1], y0, y0 + tile)
    x2 = np.clip(b[:, 2], x0, x0 + tile); y2 = np.clip(b[:, 3], y0, y0 + tile)
    w, h = np.clip(x2 - x1, 0, None), np.clip(y2 - y1, 0, None)
    area_clip = w * h
    area_orig = np.clip((b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1]), 1e-9, None)
    keep = (area_clip / area_orig >= min_vis) & (w >= min_size) & (h >= min_size)
    idx = np.where(keep)[0]
    out = np.stack([x1[idx] - x0, y1[idx] - y0, x2[idx] - x0, y2[idx] - y0], 1) if len(idx) else np.zeros((0, 4))
    return out, idx.tolist()


def nms(boxes, scores, iou_thr=0.5):
    """Plain greedy NMS. boxes xyxy."""
    if len(boxes) == 0:
        return []
    b = np.asarray(boxes, float); s = np.asarray(scores, float)
    x1, y1, x2, y2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    area = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    order = s.argsort()[::-1]; keep = []
    while order.size:
        i = order[0]; keep.append(int(i))
        if order.size == 1: break
        xx1 = np.maximum(x1[i], x1[order[1:]]); yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]]); yy2 = np.minimum(y2[i], y2[order[1:]])
        inter = np.clip(xx2 - xx1, 0, None) * np.clip(yy2 - yy1, 0, None)
        iou = inter / np.clip(area[i] + area[order[1:]] - inter, 1e-9, None)
        order = order[1:][iou <= iou_thr]
    return keep


def edge_penalty(local_boxes, tile, margin=16, factor=0.85):
    """Multiplier that demotes detections touching a tile border. The same object
    is usually complete in the overlapping neighbour tile, so NMS should prefer
    that copy. Mirrors flatbug's EDGE_CASE_MARGIN idea."""
    if len(local_boxes) == 0:
        return np.zeros(0)
    b = np.asarray(local_boxes, float)
    touch = ((b[:, 0] <= margin) | (b[:, 1] <= margin) |
             (b[:, 2] >= tile - margin) | (b[:, 3] >= tile - margin))
    return np.where(touch, factor, 1.0)


def merge_tiles(preds, iou_thr=0.5, per_class=False):
    """preds: list of (boxes_global Nx4, scores N, classes N) from each tile.
    -> merged (boxes, scores, classes) after NMS in full-image coordinates."""
    B = [p[0] for p in preds if len(p[0])]
    if not B:
        return np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)
    boxes = np.concatenate(B)
    scores = np.concatenate([p[1] for p in preds if len(p[0])])
    classes = np.concatenate([p[2] for p in preds if len(p[0])]).astype(int)
    keep_all = []
    for c in (np.unique(classes) if per_class else [None]):
        m = np.where(classes == c)[0] if c is not None else np.arange(len(boxes))
        k = nms(boxes[m], scores[m], iou_thr)
        keep_all += m[k].tolist()
    keep_all = np.array(sorted(keep_all), int)
    return boxes[keep_all], scores[keep_all], classes[keep_all]


## Step 1 — export the tiled dataset *(run once)*


In [ ]:
# ============================================================
# STEP 1 — export the tiled COCO dataset  (run once; ~30-60 min)
# ============================================================
import json, shutil
from PIL import Image
from tqdm.auto import tqdm

FORCE_REEXPORT = False     # True to rebuild from scratch

def export_split(split):
    src_dir = SRC_COCO / split
    dst_dir = TILED_COCO / split
    ann = json.load(open(src_dir / "_annotations.coco.json"))
    cats = sorted(ann["categories"], key=lambda c: c["id"])
    by_img = {}
    for a in ann["annotations"]:
        by_img.setdefault(a["image_id"], []).append(a)

    dst_dir.mkdir(parents=True, exist_ok=True)
    images, annotations = [], []
    iid = aid = 1
    n_drop = 0
    for im in tqdm(ann["images"], desc=f"tiling {split}"):
        p = src_dir / im["file_name"]
        if not p.exists():
            continue
        W, H = im["width"], im["height"]
        recs = by_img.get(im["id"], [])
        boxes = np.array([[a["bbox"][0], a["bbox"][1],
                           a["bbox"][0]+a["bbox"][2], a["bbox"][1]+a["bbox"][3]] for a in recs],
                         float).reshape(-1, 4)
        cids  = [a["category_id"] for a in recs]
        xs, ys = tile_grid(W, H, TILE, OVERLAP)
        img = Image.open(p).convert("RGB")
        for y0 in ys:
            for x0 in xs:
                lb, idx = tile_boxes(boxes, x0, y0, TILE, W, H, MIN_VIS, MIN_SIZE)
                if len(idx) == 0:
                    n_drop += 1
                    continue                      # skip empty tiles
                name = f"{Path(im['file_name']).stem}_x{x0}_y{y0}.jpg"
                img.crop((x0, y0, min(x0+TILE, W), min(y0+TILE, H))).save(
                    dst_dir / name, "JPEG", quality=92)
                tw, th = min(TILE, W-x0), min(TILE, H-y0)
                images.append({"id": iid, "file_name": name, "width": tw, "height": th})
                for b, j in zip(lb, idx):
                    annotations.append({"id": aid, "image_id": iid,
                                        "category_id": cids[j],
                                        "bbox": [float(b[0]), float(b[1]),
                                                 float(b[2]-b[0]), float(b[3]-b[1])],
                                        "area": float((b[2]-b[0])*(b[3]-b[1])), "iscrowd": 0})
                    aid += 1
                iid += 1
                if MAX_TILES_PER_SPLIT and len(images) >= MAX_TILES_PER_SPLIT:
                    break
            if MAX_TILES_PER_SPLIT and len(images) >= MAX_TILES_PER_SPLIT:
                break
        if MAX_TILES_PER_SPLIT and len(images) >= MAX_TILES_PER_SPLIT:
            break
    json.dump({"info": {"description": f"tiled {split} ({TILE}px, overlap {OVERLAP})"},
               "categories": cats, "images": images, "annotations": annotations},
              open(dst_dir / "_annotations.coco.json", "w"))
    return len(images), len(annotations), n_drop

def _tiles_are_current():
    """Is the tiled dataset on disk the one THIS config would produce?

    An existence check is not enough. The September re-run exists precisely because
    the August ground truth was wrong — boxes 1.295x too large, trays at 8.26 px/mm,
    23 classes. A leftover tiled export from then would be silently reused here and
    would invalidate both this run and the flatbug tiled run that reads the same
    directory, with nothing in the output to say so. So verify it matches.
    """
    f = TILED_COCO / "train" / "_annotations.coco.json"
    if not f.exists():
        return False, "no train annotations"
    d = json.load(open(f))
    src = json.load(open(SRC_COCO / "train" / "_annotations.coco.json"))
    got = [(c["id"], c["name"]) for c in sorted(d["categories"], key=lambda c: c["id"])]
    want = [(c["id"], c["name"]) for c in sorted(src["categories"], key=lambda c: c["id"])]
    if got != want:
        return False, f"categories differ ({len(got)} vs {len(want)} classes)"
    if not d.get("images"):
        return False, "no images"
    w = {i["width"] for i in d["images"][:200]}
    if max(w) != TILE:
        return False, f"tiles are {max(w)} px, config says {TILE}"
    return True, f"{len(d['images'])} tiles, {len(got)} classes, {TILE} px"

if TILED_COCO.exists() and not FORCE_REEXPORT and \
   (TILED_COCO / "train" / "_annotations.coco.json").exists():
    _ok, _why = _tiles_are_current()
    assert _ok, (
        f"{TILED_COCO} exists but does NOT match this config: {_why}.\n"
        f"Reusing it would train on the wrong data silently. Delete it, or set "
        f"FORCE_REEXPORT = True to rebuild.")
    print(f"{TILED_COCO} already exists and matches this config ({_why}) - skipping export")
else:
    for sp in ["train", "valid", "test"]:
        if (SRC_COCO / sp / "_annotations.coco.json").exists():
            ni, na, nd = export_split(sp)
            print(f"  {sp:5s}: {ni:7d} tiles, {na:8d} boxes  ({nd} empty tiles skipped)")


In [ ]:
# ── STEP 2 — verify the tiled dataset ──
import json
for sp in ["train", "valid", "test"]:
    f = TILED_COCO / sp / "_annotations.coco.json"
    if not f.exists():
        print(f"  {sp}: MISSING"); continue
    a = json.load(open(f))
    src = json.load(open(SRC_COCO / sp / "_annotations.coco.json"))
    n_img_files = len(list((TILED_COCO / sp).glob("*.jpg")))
    print(f"  {sp:5s}: {len(a['images']):7d} tiles ({n_img_files} files on disk) | "
          f"{len(a['annotations']):8d} boxes | src {len(src['images'])} trays, "
          f"{len(src['annotations'])} boxes | ratio {len(a['annotations'])/max(len(src['annotations']),1):.2f}x")
    assert len(a["categories"]) == len(CLASS_NAMES), "category count mismatch!"
print("\nBox ratio >1 is expected: an insect in an overlap region appears in 2 tiles.")
print("RF-DETR reads the class count from train/_annotations.coco.json ->",
      len(json.load(open(TILED_COCO/'train'/'_annotations.coco.json'))["categories"]), "classes")


## Step 3 — train on tiles


In [ ]:
# ============================================================
# STEP 3 — train RF-DETR on the tiles (no resize: tile == RESOLUTION)
# ============================================================
import sys
from pathlib import Path
_src = str(Path(".").resolve() / "rf-detr-1.7.1" / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)
from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRLarge
ModelClass = {"RFDETRNano": RFDETRNano, "RFDETRSmall": RFDETRSmall,
              "RFDETRMedium": RFDETRMedium, "RFDETRLarge": RFDETRLarge}[MODEL_VARIANT]

model = ModelClass(resolution=RESOLUTION, num_classes=len(CLASS_NAMES))
assert model.model_config.num_classes == len(CLASS_NAMES)
print(f"num_classes = {model.model_config.num_classes} | resolution {RESOLUTION} "
      f"== TILE {TILE} -> tiles are fed 1:1, no downscaling")

model.train(
    dataset_dir=str(TILED_COCO), output_dir=OUTPUT_DIR, class_names=CLASS_NAMES,
    epochs=EPOCHS, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
    lr=LR, lr_encoder=LR_ENCODER, lr_drop=LR_DROP, warmup_epochs=WARMUP_EPOCHS,
    weight_decay=WEIGHT_DECAY, multi_scale=MULTI_SCALE, use_ema=USE_EMA,
    checkpoint_interval=CHECKPOINT_INTERVAL, eval_interval=EVAL_INTERVAL, run_test=RUN_TEST,
    early_stopping=EARLY_STOPPING, early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA, progress_bar=PROGRESS_BAR,
    tensorboard=TENSORBOARD, wandb=WANDB, num_workers=NUM_WORKERS,
    devices=NUM_GPUS, strategy=DDP_STRATEGY, resume=None,
)
print(f"\ncheckpoints -> {OUTPUT_DIR}")
print("NOTE: the test mAP printed above is on TILES. The number comparable to your "
      "other runs is produced by the tiled-inference cell below (full trays).")


## Step 4 — tiled inference on full trays, then merge

**RESUMING after training?** Run only: **CONFIG** -> **tiling core** -> this cell -> the scoring cell. Skip Step 1 (the tiled dataset already exists) and Step 3 (training is done); this cell loads `checkpoint_best_total.pth` from `OUTPUT_DIR` itself.

The test mAP printed by training is on **tiles** and is not comparable to your other runs. This is.


In [ ]:
# ============================================================
# STEP 4 — tiled inference on FULL trays + merge  (comparable to the other runs)
# ============================================================
import sys, json, time
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

# self-contained: rfdetr lives in the source tree, not site-packages, so the path
# insert must happen here too (the training cell above may be skipped on a resume)
_src = str(Path(".").resolve() / "rf-detr-1.7.1" / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)
from rfdetr import RFDETR

EVAL_SPLIT = "test"
CKPT = Path(OUTPUT_DIR) / "checkpoint_best_total.pth"
assert CKPT.exists(), f"missing {CKPT}"
m = RFDETR.from_checkpoint(str(CKPT), resolution=RESOLUTION)  # from_checkpoint does NOT restore resolution (falls back to the 704 default)

src_dir = SRC_COCO / EVAL_SPLIT
ann = json.load(open(src_dir / "_annotations.coco.json"))
cats = sorted(ann["categories"], key=lambda c: c["id"])
name_to_idx = {c["name"]: i for i, c in enumerate(cats)}
catid_to_idx = {c["id"]: i for i, c in enumerate(cats)}
GT = {}
for a in ann["annotations"]:
    b = a["bbox"]
    GT.setdefault(a["image_id"], []).append(
        (catid_to_idx[a["category_id"]], b[0], b[1], b[0]+b[2], b[1]+b[3]))

MAX_EVAL_IMAGES = None          # None = whole split
metas = ann["images"] if MAX_EVAL_IMAGES is None else ann["images"][:MAX_EVAL_IMAGES]

preds_out, t0 = {}, time.time()
for im in tqdm(metas, desc="tiled inference"):
    p = src_dir / im["file_name"]
    if not p.exists(): continue
    W, H = im["width"], im["height"]
    img = Image.open(p).convert("RGB")
    xs, ys = tile_grid(W, H, TILE, OVERLAP)
    tile_preds = []
    for y0 in ys:
        for x0 in xs:
            crop = img.crop((x0, y0, min(x0+TILE, W), min(y0+TILE, H)))
            d = m.predict(crop, threshold=INFER_CONF)
            if d is None or len(d) == 0: continue
            lb = np.asarray(d.xyxy, float).reshape(-1, 4)
            sc = np.asarray(d.confidence, float)
            nm = d.data.get("class_name", None)
            cl = (np.array([name_to_idx.get(str(n), 0) for n in nm]) if nm is not None
                  and len(nm) == len(sc) else np.asarray(d.class_id, int))
            sc = sc * edge_penalty(lb, TILE, EDGE_MARGIN, EDGE_FACTOR)
            tile_preds.append((lb + np.array([x0, y0, x0, y0]), sc, cl))
    # CLASS-AGNOSTIC. per_class=True let the same insect, detected as two different
    # classes in two overlapping tiles, survive as two boxes — NMS never compared
    # them. Measured on the August predictions: 31% of RF-DETR tiled's detections
    # were co-located pairs with different labels. Agnostic NMS keeps the best label.
    mb, ms, mc = merge_tiles(tile_preds, iou_thr=MERGE_IOU, per_class=False)
    preds_out[im["id"]] = {"boxes": mb.tolist(), "scores": ms.tolist(), "classes": mc.tolist()}

json.dump(preds_out, open(Path(OUTPUT_DIR) / f"tiled_preds_{EVAL_SPLIT}.json", "w"))
print(f"\n{sum(len(v['scores']) for v in preds_out.values())} merged detections over "
      f"{len(preds_out)} full trays in {time.time()-t0:.0f}s")


In [ ]:
# ── STEP 5 — score the merged full-tray predictions (COCO 101-pt AP) ──
import json, numpy as np, pandas as pd
from pathlib import Path

# ── self-contained: rebuild GT / CLASS_NAMES from the COCO annotations when the
# inference cell above was skipped (e.g. re-scoring an already-finished run) ──
EVAL_SPLIT = globals().get("EVAL_SPLIT", "test")
if "GT" not in globals() or "CLASS_NAMES" not in globals():
    _ann = json.load(open(Path(SRC_COCO)/EVAL_SPLIT/"_annotations.coco.json"))
    _cats = sorted(_ann["categories"], key=lambda c: c["id"])
    CLASS_NAMES = [c["name"] for c in _cats]
    _c2i = {c["id"]: i for i, c in enumerate(_cats)}
    GT = {}
    for _a in _ann["annotations"]:
        _b = _a["bbox"]
        GT.setdefault(_a["image_id"], []).append(
            (_c2i[_a["category_id"]], _b[0], _b[1], _b[0]+_b[2], _b[1]+_b[3]))
    print(f"rebuilt GT: {len(GT)} images, {len(_ann['annotations'])} boxes, "
          f"{len(CLASS_NAMES)} classes (split={EVAL_SPLIT})")
IOU_VEC = np.linspace(0.5, 0.95, 10)
NC = len(CLASS_NAMES)

def iou_mat(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    a, b = np.asarray(a, float), np.asarray(b, float)
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb-lt, 0, None); inter = wh[..., 0]*wh[..., 1]
    aa = (a[:, 2]-a[:, 0])*(a[:, 3]-a[:, 1]); ab = (b[:, 2]-b[:, 0])*(b[:, 3]-b[:, 1])
    return inter/np.clip(aa[:, None]+ab[None]-inter, 1e-9, None)

# NumPy 2.0 removed np.trapz in favour of np.trapezoid. The training envs run
# different NumPy majors (flatbug torch 2.13 / rfdetr torch 2.9), so bind once
# and use _TRAPZ rather than depending on which one is installed.
_TRAPZ = getattr(np, "trapezoid", None) or np.trapz
def _ap(rec, prec):
    mrec = np.concatenate(([0.], rec, [1.])); mpre = np.concatenate(([1.], prec, [0.]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101); return float(_TRAPZ(np.interp(x, mrec, mpre), x))

if "preds_out" not in globals():          # re-score without re-running inference
    _f = Path(OUTPUT_DIR)/f"tiled_preds_{EVAL_SPLIT}.json"
    preds_out = {int(k): v for k, v in json.load(open(_f)).items()}
    print(f"loaded {len(preds_out)} images from {_f}")

tps, confs, pcls, tcls = [], [], [], []
for iid, pr in preds_out.items():
    g = GT.get(iid, [])
    gb = np.array([x[1:] for x in g]).reshape(-1, 4); gc = np.array([x[0] for x in g], int)
    pb = np.array(pr["boxes"]).reshape(-1, 4); ps = np.array(pr["scores"]); pc = np.array(pr["classes"], int)
    tp = np.zeros((len(pb), len(IOU_VEC)), bool)
    if len(pb) and len(gb):
        M = iou_mat(pb, gb); same = pc[:, None] == gc[None, :]
        for ti, thr in enumerate(IOU_VEC):
            ci, gi = np.where((M >= thr) & same)
            if not len(ci): continue
            o = np.argsort(-M[ci, gi]); up, ug = set(), set()
            for c, gg in zip(ci[o], gi[o]):
                if c in up or gg in ug: continue
                up.add(c); ug.add(gg); tp[c, ti] = True
    tps.append(tp); confs.append(ps); pcls.append(pc); tcls.append(gc)

tp = np.concatenate(tps); conf = np.concatenate(confs)
pcls = np.concatenate(pcls); tcls = np.concatenate(tcls)
o = np.argsort(-conf); tp, conf, pcls = tp[o], conf[o], pcls[o]
n_gt = np.bincount(tcls, minlength=NC)

rows, ap50, ap75, ap5095 = [], [], [], []
precs, recs = [], []
for c in range(NC):
    i = pcls == c
    if n_gt[c] == 0 or i.sum() == 0:
        rows.append({"class": CLASS_NAMES[c], "instances": int(n_gt[c]), "AP50": 0.0, "AP75": 0.0,
                     "AP50_95": 0.0, "precision": 0.0, "recall": 0.0, "F1": 0.0, "conf_at_bestF1": 0.0})
        ap50.append(0.0); ap75.append(0.0); ap5095.append(0.0)
        precs.append(0.0); recs.append(0.0); continue
    tpc = tp[i].cumsum(0); fpc = (1-tp[i]).cumsum(0)
    rec = tpc/(n_gt[c]+1e-16); prec = tpc/(tpc+fpc+1e-16)
    aps = [_ap(rec[:, t], prec[:, t]) for t in range(len(IOU_VEC))]
    ap50.append(aps[0]); ap75.append(aps[5]); ap5095.append(float(np.mean(aps)))
    # Precision and recall at the best-F1 operating point, IoU 0.5 — the same
    # convention Ultralytics reports, so the tiled and whole-tray rows of the
    # comparison are directly comparable. These curves were already being computed
    # and thrown away after integrating AP.
    _p50, _r50 = prec[:, 0], rec[:, 0]
    _f1 = 2*_p50*_r50/np.clip(_p50+_r50, 1e-16, None)
    _b = int(np.argmax(_f1))
    precs.append(float(_p50[_b])); recs.append(float(_r50[_b]))
    rows.append({"class": CLASS_NAMES[c], "instances": int(n_gt[c]),
                 "AP50": round(aps[0], 4), "AP75": round(aps[5], 4),
                 "AP50_95": round(float(np.mean(aps)), 4),
                 "precision": round(float(_p50[_b]), 4),
                 "recall": round(float(_r50[_b]), 4),
                 "F1": round(float(_f1[_b]), 4),
                 "conf_at_bestF1": round(float(conf[i][_b]), 4)})
df = pd.DataFrame(rows)
df.to_csv(Path(OUTPUT_DIR)/"tiled_per_class.csv", index=False)
print(df.to_string(index=False))
print(f"\n=== TILED RF-DETR, full trays ({EVAL_SPLIT}) ===")
print(f"  mAP@50:95 : {np.mean(ap5095):.4f}")
print(f"  mAP@50    : {np.mean(ap50):.4f}")
print(f"  mAP@75    : {np.mean(ap75):.4f}   <- the metric this experiment is about")
print(f"  precision : {np.mean(precs):.4f}   (class mean at best-F1, IoU 0.5)")
print(f"  recall    : {np.mean(recs):.4f}")
print( "  compare against: RF-DETR 800px single-stage 0.3476 (@50 0.6408, @75 0.3400)")
print( "                   flatbug-L                0.4189 (@50 0.6265, @75 0.4681)")
